# Punto 3 – Activación con IA

Base de conocimiento (3.1) y system prompts hiper-personalizados (3.2) para el Agente de IA.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

RUTA_PROYECTO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RUTA_PROYECTO))

from src.data_loader import RUTA_PROCESSED, RUTA_RAW
from src.knowledge_base import RUTA_KB, construir_base, guardar_base
from src.ubicacion import categorias_top
from src.utils import imprimir_checklist

pd.set_option("display.width", 200)
items = pd.read_parquet(RUTA_PROCESSED / "items_enriquecidos.parquet")  # pedidos entregados (1.1)
experiencia = pd.read_parquet(RUTA_PROCESSED / "pedidos_experiencia.parquet")  # 1.2
segmentados = pd.read_parquet(RUTA_PROCESSED / "clientes_segmentados.parquet")  # 2.1

# Punto 3.1 – Base de conocimiento

**Selección:** el producto con **más ingresos** de tres categorías top del 1.1: Belleza y salud (líder en ingresos), Cama, mesa y baño (líder en volumen) y Muebles y decoración. En Cama y Muebles, ese producto es además el más vendido en unidades de su categoría.

**¿Qué es cada producto?** El dataset no trae nombres, solo un `product_id` y su categoría. Leyendo las **reseñas de sus compradores** se identificó el tipo de producto:
- Belleza y salud → un **adipómetro** ("vai me ajudar muito na prática clínica").
- Cama, mesa y baño → un **juego de toallas** estampado y bordado.
- Muebles y decoración → un **perchero de ropa con ruedas** ("arara de roupas").

**Qué es real y qué es simulado:** precios, ventas, reseñas, entrega y segmentos son **reales**. Nombre comercial, descripción, beneficios y argumentos de venta son **simulados**, sin marcas ni especificaciones inventadas. Las quejas reales de las reseñas quedan como `nota_para_el_agente`, para que el agente no prometa de más.

In [2]:
base = construir_base(items, experiencia, segmentados)
guardar_base(base)

resumen = pd.DataFrame([{
    "producto (simulado)": p["nombre_comercial"].replace(" (simulado)", ""),
    "categoría": p["categoria"]["es"],
    "precio prom. R$": p["precio_promedio_real"],
    "p25-p75 R$": f"{p['rango_precio_p25_p75'][0]} - {p['rango_precio_p25_p75'][1]}",
    "unidades": p["unidades_vendidas"],
    "ingresos R$": p["ingresos_totales"],
    "reseña": p["score_promedio_resenas"],
    "días entrega": p["tiempo_entrega_promedio_dias"],
    "% a tiempo": p["pct_entregas_a_tiempo"],
    "segmento con mayor afinidad": p["segmentos_objetivo"][0]["segmento"],
} for p in base["productos"]])
print(f"Guardado: {RUTA_KB}")
resumen

Guardado: C:\Users\user\Downloads\prueba-tecnica-proteccion-ds-ia\agent\knowledge_base.json


,producto (simulado),categoría,precio prom. R$,p25-p75 R$,unidades,ingresos R$,reseña,días entrega,% a tiempo,segmento con mayor afinidad
0,Adipómetro Profesional,Belleza y salud,327.63,325.0 - 330.0,194,63560.00,4.23,13.2,84.9,Fieles
1,Juego de Toallas Jardín Bordado,"Cama, mesa y baño",88.15,86.9 - 89.9,477,42049.66,3.93,13.1,93.6,Fieles
2,Perchero Organizador con Ruedas,Muebles y decoración,71.35,69.9 - 69.9,520,37104.30,4.13,11.6,92.9,Fieles


Así se ve un producto dentro del JSON, que es lo que recibirá el agente como `[BASE_CONOCIMIENTO]`:

In [3]:
print(json.dumps(base["productos"][0], ensure_ascii=False, indent=2))

{
  "id": "bb50f2e236e5eea0100680137654686c",
  "nombre_comercial": "Adipómetro Profesional (simulado)",
  "identificado_por": "Reseñas: 'Adipômetro... vai me ajudar muito na prática clínica'",
  "categoria": {
    "es": "Belleza y salud",
    "en": "health_beauty",
    "pt": "beleza_saude"
  },
  "descripcion": "Para quien se toma en serio el bienestar: mide la grasa corporal como lo hacen nutricionistas y entrenadores. Liviano, fácil de llevar a consulta y hecho para acompañar cada evaluación de tus pacientes o de tu propio progreso.",
  "beneficios_clave": [
    "Seguimiento real del progreso, no solo del peso",
    "Portátil y liviano",
    "Pensado para uso profesional"
  ],
  "precio_promedio_real": 327.63,
  "precio_mediano": 330.0,
  "rango_precio_p25_p75": [
    325.0,
    330.0
  ],
  "flete_promedio": 19.06,
  "unidades_vendidas": 194,
  "ingresos_totales": 63560.0,
  "score_promedio_resenas": 4.23,
  "tiempo_entrega_promedio_dias": 13.2,
  "pct_entregas_a_tiempo": 84.9,
  "

### Validaciones

In [4]:
# Precio recalculado de forma independiente desde los CSV originales (pedidos entregados)
items_raw = pd.read_csv(RUTA_RAW / "olist_order_items_dataset.csv")
pedidos_raw = pd.read_csv(RUTA_RAW / "olist_orders_dataset.csv")
entregados = items_raw[items_raw["order_id"].isin(pedidos_raw.loc[pedidos_raw["order_status"] == "delivered", "order_id"])]
precio_raw = entregados.groupby("product_id")["price"].mean().round(2)

top_11 = categorias_top(items)
kb = json.loads(RUTA_KB.read_text(encoding="utf-8"))  # JSON válido si carga sin error
campos = ["id", "nombre_comercial", "categoria", "descripcion", "beneficios_clave", "precio_promedio_real",
          "precio_mediano", "rango_precio_p25_p75", "flete_promedio", "unidades_vendidas", "score_promedio_resenas",
          "tiempo_entrega_promedio_dias", "pct_entregas_a_tiempo", "segmentos_objetivo", "argumentos_venta_por_perfil"]
diferencias = [abs(p["precio_promedio_real"] - precio_raw[p["id"]]) for p in kb["productos"]]

validaciones = [
    ("Los 3 product_id existen en el dataset", all(p["id"] in precio_raw.index for p in kb["productos"]), "3 de 3"),
    ("Los 3 pertenecen a categorías top del 1.1", all(p["categoria"]["es"] in top_11 for p in kb["productos"]),
     ", ".join(p["categoria"]["es"] for p in kb["productos"])),
    ("Precio promedio recalculado desde los CSV = JSON", max(diferencias) == 0, f"diferencia máxima {max(diferencias)}"),
    ("JSON válido con 3 entradas completas", len(kb["productos"]) == 3 and all(all(c in p for c in campos) for p in kb["productos"]),
     f"{len(kb['productos'])} productos x {len(campos)} campos"),
]
imprimir_checklist(validaciones)

CHECKLIST DE VALIDACIÓN
----------------------------------------------------------------------
✅ Los 3 product_id existen en el dataset: 3 de 3
✅ Los 3 pertenecen a categorías top del 1.1: Belleza y salud, Cama, mesa y baño, Muebles y decoración
✅ Precio promedio recalculado desde los CSV = JSON: diferencia máxima 0.0
✅ JSON válido con 3 entradas completas: 3 productos x 15 campos
----------------------------------------------------------------------
TODAS LAS VALIDACIONES PASARON


True

### Conclusión del 3.1

- **Tres productos reales con cifras reales:** Adipómetro (R$ 327,63), Juego de toallas (R$ 88,15) y Perchero con ruedas (R$ 71,35). Cubren tres niveles de precio y tres perfiles de cliente.
- **Las reseñas le dieron nombre a los productos:** donde el dataset solo trae un código, los comentarios de los compradores revelan qué se vendió. Así los textos son simulados, pero no inventados de la nada.
- **La base también advierte:** las quejas reales (toallas delgadas, pedidos incompletos, envíos no entregados) quedan como notas para que el agente no prometa lo que el producto no cumple. Eso es clave para el cliente con mala experiencia del 3.2.
- **Segmentos:** el Adipómetro tiene afinidad con clientes Fieles y VIP (índice 1,5 y 1,4); toallas y perchero, con Fieles.

# Punto 3.2 – System prompts hiper-personalizados

Tres plantillas en `agent/prompts/` (joven digital, mayor con mala experiencia y VIP). Todas tienen la misma estructura: rol, contexto con los 5 placeholders, objetivo, tono, estrategia de recomendación, guardarraíles, formato y ejemplo de primer mensaje. Lo que cambia es **el tono y la forma de recomendar**.

`construir_prompt()` las llena con datos reales:
- `[PERFIL_DE_CLIENTE]` = segmento y subsegmento del 2.1.
- `[HISTORIAL_COMPRAS]` = resumen en texto calculado en el 2.1 (pedidos, gasto, ticket, categoría favorita, recencia, reseña y mala experiencia con su motivo del 1.2).
- `[BASE_CONOCIMIENTO]` = el JSON del 3.1.
- `[EDAD]` y `[GÉNERO]` = simulados, como si vinieran del CRM.

In [5]:
from src.agent_context import ESCENARIOS, PLACEHOLDERS, RUTA_PROMPTS, construir_prompt, elegir_clientes

# Edad y género simulados (CRM): solo modulan el tono, nunca los gustos
CRM = {"joven_digital": (24, "Mujer"), "mayor_mala_experiencia": (67, "Hombre"), "vip": (45, "Mujer")}

elegidos = elegir_clientes(segmentados)
ficha = segmentados.set_index("customer_unique_id")
carpeta = RUTA_PROMPTS / "ejemplos"
carpeta.mkdir(exist_ok=True)

escenarios_demo = {}
for escenario, cid in elegidos.items():
    edad, genero = CRM[escenario]
    prompt = construir_prompt(escenario, cid, edad, genero, segmentados)
    (carpeta / f"{escenario}.md").write_text(prompt, encoding="utf-8")
    escenarios_demo[escenario] = {"customer_unique_id": cid, "edad": edad, "genero": genero,
                                  "segmento": ficha.loc[cid, "segmento"], "historial": ficha.loc[cid, "historial_compras"],
                                  "ciudad": ficha.loc[cid, "ciudad"].title(), "uf": ficha.loc[cid, "estado"], "system_prompt": prompt}
    print(f"{escenario:24} {cid}\n   {ficha.loc[cid, 'historial_compras']}\n")

# Insumo del chatbot desplegado (parte B)
(carpeta / "escenarios.json").write_text(json.dumps(escenarios_demo, ensure_ascii=False, indent=2), encoding="utf-8")

joven_digital            1373e04979cfa0fb2092909abbd57f25
   3 pedidos | gasto total R$609 | ticket prom. R$203 | categoría favorita: Moda: bolsos y accesorios | última compra hace 118 días | score prom. 5.0 | mala experiencia: no | segmento: Fieles



mayor_mala_experiencia   15823b5826557e80b07c3d5dc611568e
   1 pedido | gasto total R$44 | ticket prom. R$44 | categoría favorita: Mascotas | última compra hace 354 días | score prom. 1.0 | mala experiencia: sí | motivo: entrega / retraso | segmento: Perdidos / Inactivos

vip                      e36d108281c19ba00d15de8880a51b69
   1 pedido | gasto total R$1,211 | ticket prom. R$1,211 | categoría favorita: Informática y accesorios | última compra hace 33 días | score prom. 5.0 | mala experiencia: no | segmento: VIP / Alto valor



24936

Así llega al LLM el prompt del cliente mayor con mala experiencia (primeras líneas):

In [6]:
print(escenarios_demo["mayor_mala_experiencia"]["system_prompt"][:1400])

# System prompt – Cliente mayor, conservador o inactivo, con una mala experiencia previa

## 1. Rol e identidad
Eres **Andrés**, asesor de servicio al cliente de la tienda en línea. Tu prioridad es escuchar, recuperar la confianza del cliente y, solo después, ayudarle a comprar con tranquilidad.

## 2. Contexto del cliente
- Edad: 67 años (dato del CRM)
- Género: Hombre (dato del CRM)
- Perfil de cliente (segmento): Perdidos / Inactivos (Perdido con mala experiencia)
- Historial de compras: 1 pedido | gasto total R$44 | ticket prom. R$44 | categoría favorita: Mascotas | última compra hace 354 días | score prom. 1.0 | mala experiencia: sí | motivo: entrega / retraso | segmento: Perdidos / Inactivos
- Base de conocimiento (únicos productos que puedes ofrecer):
[{"id":"bb50f2e236e5eea0100680137654686c","nombre_comercial":"Adipómetro Profesional (simulado)","identificado_por":"Reseñas: 'Adipômetro... vai me ajudar muito na prática clínica'","categoria":{"es":"Belleza y salud","en":"health_

### Validaciones

In [7]:
plantillas = {e: (RUTA_PROMPTS / f"{e}.md").read_text(encoding="utf-8") for e in ESCENARIOS}
llenos = {e: d["system_prompt"] for e, d in escenarios_demo.items()}
tokens = {e: round(len(p) / 4) for e, p in llenos.items()}  # aprox. 4 caracteres por token
perfil_ok = (ficha.loc[elegidos["joven_digital"], "segmento"] == "Fieles"
             and ficha.loc[elegidos["mayor_mala_experiencia"], "tuvo_mala_experiencia"]
             and ficha.loc[elegidos["vip"], "segmento"] == "VIP / Alto valor")
validaciones = [
    ("Cada plantilla tiene los 5 placeholders exactos", all(all(p in t for p in PLACEHOLDERS) for t in plantillas.values()), "3 de 3"),
    ("Ningún prompt lleno conserva placeholders", not any(p in t for t in llenos.values() for p in PLACEHOLDERS), "0 sin reemplazar"),
    ("Los clientes cumplen el perfil", bool(perfil_ok), "Fiel frecuente · inactivo con mala experiencia (retraso) · VIP"),
    ("Longitud razonable (< 4.000 tokens)", max(tokens.values()) < 4000, str(tokens)),
]
imprimir_checklist(validaciones)

CHECKLIST DE VALIDACIÓN
----------------------------------------------------------------------
✅ Cada plantilla tiene los 5 placeholders exactos: 3 de 3
✅ Ningún prompt lleno conserva placeholders: 0 sin reemplazar
✅ Los clientes cumplen el perfil: Fiel frecuente · inactivo con mala experiencia (retraso) · VIP
✅ Longitud razonable (< 4.000 tokens): {'joven_digital': 1818, 'mayor_mala_experiencia': 1926, 'vip': 1900}
----------------------------------------------------------------------
TODAS LAS VALIDACIONES PASARON


True

### Conclusión del 3.2

- **Un mismo agente, tres personalidades:** Sofi (joven digital: tutea, mensajes cortos, emojis y llamada a la acción), Andrés (mayor con mala experiencia: usted, primero disculpa y escucha, luego productos que llegan a tiempo) y Valentina (VIP: consultiva, pregunta antes de recomendar y propone cantidades con precios reales).
- **El dolor del 1.2 cambia la conversación:** con el cliente que sufrió un retraso, el agente no vende en el primer mensaje, prioriza el `pct_entregas_a_tiempo` y descarta el Adipómetro, que es el que peor entrega.
- **Guardarraíles comunes:** solo productos de la base, nada de precios o descuentos inventados, edad y género solo para el tono, sin revelar datos internos y con escalamiento a un asesor humano.
- **Demo:** los tres prompts llenos alimentan el chatbot desplegado en AWS (Claude en Amazon Bedrock).